# DV90 CNA Benchmarking – Report Generator (v3, all tools PCR-corrected)
Run all cells in order.

In [1]:
import pandas as pd
from datetime import date
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Image,
    Table, TableStyle, PageBreak, HRFlowable
)
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_JUSTIFY

RESULTS_DIR = '/sharedFolder/Results'
PERF_CSV    = f'{RESULTS_DIR}/performance_vs_CCLE_DV90.csv'
COMP_CSV    = f'{RESULTS_DIR}/CNV_comparison_table_DV90.csv'
CONF_PNG    = f'{RESULTS_DIR}/confusion_matrix_DV90.png'
COMP_PNG    = f'{RESULTS_DIR}/comparison_3tools_DV90.png'
PDF_OUT     = f'{RESULTS_DIR}/DV90_CNA_Benchmarking_Report_v3.pdf'

perf = pd.read_csv(PERF_CSV)
comp = pd.read_csv(COMP_CSV, index_col=0)
today = date.today().strftime('%B %d, %Y')
print('Data loaded')
print(perf)

Data loaded
            Tool CNA_type   TP   FN   FP    TN  Sensitivity  Specificity
0         AtaCNV     gain  161   24  454  1721        0.870        0.791
1         AtaCNV     loss    7   22  666  1665        0.241        0.714
2  epiAneufinder     gain  161   24  912  1263        0.870        0.581
3  epiAneufinder     loss   17   12  503  1828        0.586        0.784
4        CONGASp     gain   58  127  726  1449        0.314        0.666
5        CONGASp     loss    9   20  682  1649        0.310        0.707


In [2]:
base = getSampleStyleSheet()

title_style = ParagraphStyle('title',
    fontSize=20, fontName='Helvetica-Bold',
    alignment=TA_CENTER, spaceAfter=6)
subtitle_style = ParagraphStyle('subtitle',
    fontSize=12, fontName='Helvetica',
    alignment=TA_CENTER, textColor=colors.HexColor('#444444'), spaceAfter=4)
date_style = ParagraphStyle('date',
    fontSize=9, fontName='Helvetica',
    alignment=TA_CENTER, textColor=colors.grey, spaceAfter=16)
h1_style = ParagraphStyle('h1',
    fontSize=13, fontName='Helvetica-Bold',
    spaceBefore=14, spaceAfter=6,
    textColor=colors.HexColor('#1a1a2e'))
h2_style = ParagraphStyle('h2',
    fontSize=11, fontName='Helvetica-Bold',
    spaceBefore=10, spaceAfter=4,
    textColor=colors.HexColor('#333333'))
body_style = ParagraphStyle('body',
    fontSize=10, fontName='Helvetica',
    leading=15, alignment=TA_JUSTIFY, spaceAfter=8)
caption_style = ParagraphStyle('caption',
    fontSize=9, fontName='Helvetica-Oblique',
    alignment=TA_CENTER, textColor=colors.HexColor('#555555'),
    spaceBefore=4, spaceAfter=12)
note_style = ParagraphStyle('note',
    fontSize=9, fontName='Helvetica',
    leading=13, textColor=colors.HexColor('#555555'),
    leftIndent=12, spaceAfter=6)

def HR():
    return HRFlowable(width='100%', thickness=0.8,
                      color=colors.HexColor('#cccccc'), spaceAfter=8)
def H1(text): return Paragraph(text, h1_style)
def H2(text): return Paragraph(text, h2_style)
def P(text):  return Paragraph(text, body_style)
def Note(text): return Paragraph(f'<i>{text}</i>', note_style)
def SP(n=8):  return Spacer(1, n)

print('Styles OK')

Styles OK


In [3]:
doc = SimpleDocTemplate(
    PDF_OUT, pagesize=A4,
    leftMargin=2*cm, rightMargin=2*cm,
    topMargin=2*cm, bottomMargin=2*cm
)
W = A4[0] - 4*cm
story = []

# ══════════════════════════════════════════════════════════
# PAGE 1 – Title + Background + Methods
# ══════════════════════════════════════════════════════════
story += [
    SP(20),
    Paragraph('DV90 scATAC-seq CNA Caller Benchmarking', title_style),
    Paragraph('Comparison of AtaCNV, epiAneufinder and CONGASp<br/>against CCLE ABSOLUTE Ground Truth', subtitle_style),
    Paragraph(f'Internal Report &nbsp;–&nbsp; {today} &nbsp;(v3, all tools PCR-corrected)', date_style),
    HR(),
]

story += [
    H1('1.  What are Copy Number Alterations (CNAs) and why do they matter?'),
    P('In a healthy human cell, every chromosome is present in two copies — one inherited '
      'from each parent. In cancer cells, this balance is often disrupted: some regions of '
      'the genome are duplicated (gaining extra copies) while others are deleted (losing one '
      'or both copies). These structural changes are called <b>Copy Number Alterations (CNAs)</b>.'),
    P('CNAs are not random noise — they are often the engine of tumour growth. A region that '
      'is amplified may carry an oncogene that drives cell proliferation. A deleted region may '
      'have lost a tumour suppressor. Knowing where CNAs occur in a tumour is therefore '
      'fundamental both for understanding the biology of the disease and for identifying '
      'potential therapeutic targets.'),
    H2('Why use scATAC-seq to detect CNAs?'),
    P('Single-cell ATAC-seq (scATAC-seq) is a technique that measures, for each individual '
      'cell, which regions of the genome are physically "open" and accessible to regulatory '
      'proteins. Its primary use is to study gene regulation and cell identity. However, it '
      'also produces a useful by-product: a count of how many DNA fragments were recovered '
      'from each region of the genome.'),
    P('The key insight is that <b>regions with more DNA copies will, on average, produce more '
      'fragments</b> — simply because there is more physical DNA to be cut and sequenced. '
      'Conversely, deleted regions will produce fewer fragments. By averaging this signal '
      'over large genomic windows (1 megabase = 1 million base pairs), we can wash out the '
      'local variation due to gene activity and isolate the underlying structural signal. '
      'This allows CNA inference without any additional experiment.'),
    P('The DV90 cell line used in this study is a lung cancer model with '
      '3,472 single cells profiled, with an average of ~60,000 unique fragments per cell.'),
]

story += [
    H1('2.  What did we do? (Methods)'),
    H2('2.1  Three tools, one dataset'),
    P('We tested three published computational tools that take scATAC-seq data as input and '
      'attempt to call CNAs. All three were run on the same DV90 fragment file and with the '
      'same genomic bin size (1 Mb) to ensure a fair comparison:'),
]

tool_data = [
    ['Tool', 'Strategy', 'GitHub'],
    ['AtaCNV', 'Counts fragments per 1 Mb bin, normalises for GC\ncontent and library size, then uses BICseq2 to\nidentify breakpoints and segment the genome.', 'XiDsLab/AtaCNV'],
    ['epiAneufinder', 'Similar binning strategy with GC correction,\nfollowed by binary segmentation to identify\nregions of gain, loss or normal copy number.', 'colomemaria/epiAneufinder'],
    ['CONGASp', 'Bayesian probabilistic model that clusters cells\nby their CNA profile and infers the most likely\ncopy number state for each cluster.', 'caravagnalab/CONGASp'],
]
t = Table(tool_data, colWidths=[3.5*cm, 9*cm, 4.5*cm])
t.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#1a1a2e')),
    ('TEXTCOLOR',  (0,0), (-1,0), colors.white),
    ('FONTNAME',   (0,0), (-1,0), 'Helvetica-Bold'),
    ('FONTSIZE',   (0,0), (-1,-1), 9),
    ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.HexColor('#f5f5f5'), colors.white]),
    ('GRID',       (0,0), (-1,-1), 0.4, colors.HexColor('#cccccc')),
    ('VALIGN',     (0,0), (-1,-1), 'TOP'),
    ('TOPPADDING', (0,0), (-1,-1), 6),
    ('BOTTOMPADDING', (0,0), (-1,-1), 6),
    ('LEFTPADDING', (0,0), (-1,-1), 6),
]))
story += [t, SP(10)]

story += [
    H2('2.2  How did we validate the results? The ground truth'),
    P('A key challenge in benchmarking CNA callers is knowing the correct answer. '
      'To do this, we used data from the <b>Cancer Cell Line Encyclopedia (CCLE)</b>, '
      'a large public database that has characterised hundreds of cancer cell lines '
      'using conventional bulk DNA sequencing — a method that directly measures '
      'copy numbers with high accuracy.'),
    P('For DV90, the CCLE provides a segmentation table produced by the ABSOLUTE algorithm, '
      'which infers the integer copy number of each genomic segment. We mapped these '
      'segments onto the same 1 Mb bins used by our tools and labelled each bin as:'),
]

cn_data = [
    ['Label', 'Copy Number', 'Meaning'],
    ['neutral', 'CN = 2', 'Normal diploid — one copy from each parent. No alteration.'],
    ['gain',    'CN > 2', 'Amplification — extra copies of this region are present.'],
    ['loss',    'CN < 2', 'Deletion — one or both copies of this region are missing.'],
]
t2 = Table(cn_data, colWidths=[3*cm, 3.5*cm, 10.5*cm])
t2.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#2d6a4f')),
    ('TEXTCOLOR',  (0,0), (-1,0), colors.white),
    ('FONTNAME',   (0,0), (-1,0), 'Helvetica-Bold'),
    ('FONTSIZE',   (0,0), (-1,-1), 9),
    ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.HexColor('#f0faf4'), colors.white]),
    ('GRID',       (0,0), (-1,-1), 0.4, colors.HexColor('#cccccc')),
    ('VALIGN',     (0,0), (-1,-1), 'MIDDLE'),
    ('TOPPADDING', (0,0), (-1,-1), 6),
    ('BOTTOMPADDING', (0,0), (-1,-1), 6),
    ('LEFTPADDING', (0,0), (-1,-1), 6),
]))
story += [t2, SP(10)]

story += [
    H2('2.3  How did we measure performance?'),
    P('For each tool, we compared its predictions against the CCLE ground truth across '
      '<b>2,360 overlapping 1 Mb bins</b>. For each bin, the tool prediction is either '
      'correct or wrong. We summarised this with two metrics:'),
    P('<b>Sensitivity</b> (also called Recall): out of all the bins that <i>truly</i> have a '
      'gain (or loss) according to CCLE, what fraction did the tool correctly identify? '
      'A high sensitivity means the tool rarely misses real alterations.'),
    P('<b>Specificity</b>: out of all the bins that are <i>truly neutral</i> according to CCLE, '
      'what fraction did the tool correctly call neutral? A high specificity means the tool '
      'rarely raises false alarms — it does not see CNAs where there are none.'),
    Note('The ideal tool would have both sensitivity and specificity close to 1.0. '
         'In practice there is always a trade-off: being more sensitive often means '
         'generating more false alarms, and vice versa.'),
    H2('2.4  A note on PCR duplicate correction'),
    P('The 10x Genomics fragment file stores, for each unique fragment, the number of times '
      'it was sequenced (column 5). In a first version of this analysis, this count was '
      'mistakenly used as a weight when building the count matrix, effectively including '
      'PCR duplicates in the fragment counts. This has been corrected: the count matrix '
      'now counts each unique fragment exactly once, regardless of how many times it was '
      'sequenced. The results reported here reflect this correction.'),
    PageBreak()
]

# ══════════════════════════════════════════════════════════
# PAGE 2 – Results: Figures
# ══════════════════════════════════════════════════════════
story += [
    H1('3.  Results'),
    H2('3.1  Genome-wide CNA profiles'),
    P('The figure below shows the CNA signal detected by each of the three tools '
      'across all 22 autosomal chromosomes. Each point represents the average signal '
      'for one chromosome. The dashed horizontal line marks the baseline (no alteration). '
      'Points above the line suggest amplification; points below suggest deletion.'),
]

img1 = Image(COMP_PNG, width=W, height=W*0.55)
story += [
    img1,
    Paragraph(
        '<b>Figure 1.</b> Chromosome-level CNA signal for the three tools. '
        'Top panel: AtaCNV copy ratio (dashed line = baseline 1.0, above = gain, below = loss). '
        'Middle panel: CONGASp CN state per clone (K=2 clones identified). '
        'Bottom panel: epiAneufinder mean CN state (0=loss, 1=normal, 2=gain).',
        caption_style),
    H2('3.2  Comparison against CCLE ground truth: confusion matrices'),
    P('The confusion matrices below show, for each tool, how its predictions '
      'match the CCLE ground truth at the level of individual 1 Mb bins. '
      'Each matrix has three rows (the true CCLE class) and three columns '
      '(what the tool predicted). The numbers on the diagonal are correct calls; '
      'off-diagonal numbers are errors.'),
    P('For example, in the AtaCNV matrix, the top-left cell (161) means that '
      '161 bins were correctly identified as gain. The cell to its right (10) '
      'means that 10 bins that were truly gain were incorrectly called loss (missed). '
      'The bottom row shows how many truly neutral bins were misclassified.'),
]

img2 = Image(CONF_PNG, width=W, height=W*0.42)
story += [
    img2,
    Paragraph(
        '<b>Figure 2.</b> Confusion matrices for AtaCNV (left), epiAneufinder (centre) '
        'and CONGASp (right) vs CCLE ABSOLUTE ground truth. '
        'Rows = true class; columns = predicted class. '
        'Numbers on the diagonal are correct calls (darker = more frequent). '
        'The large off-diagonal values in the neutral row reflect the class imbalance: '
        '91% of bins are truly neutral, making false positives common.',
        caption_style),
    PageBreak()
]

# ══════════════════════════════════════════════════════════
# PAGE 3 – Tables + Conclusions
# ══════════════════════════════════════════════════════════
story += [
    H2('3.3  Performance metrics'),
    P('The table below summarises sensitivity and specificity for each tool and '
      'each type of alteration (gain or loss). TP = true positives (correct detections), '
      'FN = false negatives (missed alterations), FP = false positives (false alarms), '
      'TN = true negatives (correctly identified neutral bins). '
      'Green highlight marks the best value in each column.'),
]

perf_header = ['Tool', 'CNA type', 'TP', 'FN', 'FP', 'TN', 'Sensitivity', 'Specificity']
perf_rows   = [[str(v) if not isinstance(v, float) else f'{v:.3f}'
                for v in row]
               for row in perf[['Tool','CNA_type','TP','FN','FP','TN',
                                 'Sensitivity','Specificity']].values.tolist()]
perf_data   = [perf_header] + perf_rows
col_w = [3.5*cm, 2.2*cm, 1.3*cm, 1.3*cm, 1.3*cm, 1.6*cm, 2.5*cm, 2.5*cm]
t3 = Table(perf_data, colWidths=col_w)

best_sens_gain = perf[perf.CNA_type=='gain']['Sensitivity'].idxmax() + 1
best_spec_gain = perf[perf.CNA_type=='gain']['Specificity'].idxmax() + 1

t3_style = [
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#1a1a2e')),
    ('TEXTCOLOR',  (0,0), (-1,0), colors.white),
    ('FONTNAME',   (0,0), (-1,0), 'Helvetica-Bold'),
    ('FONTSIZE',   (0,0), (-1,-1), 9),
    ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.HexColor('#f5f5f5'), colors.white]),
    ('GRID',       (0,0), (-1,-1), 0.4, colors.HexColor('#cccccc')),
    ('ALIGN',      (2,0), (-1,-1), 'CENTER'),
    ('VALIGN',     (0,0), (-1,-1), 'MIDDLE'),
    ('TOPPADDING', (0,0), (-1,-1), 5),
    ('BOTTOMPADDING', (0,0), (-1,-1), 5),
    ('LEFTPADDING', (0,0), (-1,-1), 5),
    ('BACKGROUND', (6, best_sens_gain), (6, best_sens_gain), colors.HexColor('#d4edda')),
    ('BACKGROUND', (7, best_spec_gain), (7, best_spec_gain), colors.HexColor('#d4edda')),
]
t3.setStyle(TableStyle(t3_style))
story += [t3, SP(14)]

story += [
    H2('3.4  Chromosome-level agreement between tools'),
    P('The table below shows, for each chromosome, what each tool called and '
      'whether the three tools agreed. "✅ 3/3" means all three tools agreed. '
      '"⚠️ 2/3" means two out of three agreed. "❌ discordante" means no clear consensus.'),
]

comp_d = comp.reset_index()
comp_d.columns = ['Chromosome', 'AtaCNV', 'CONGASp', 'epiAneufinder', 'Agreement']
comp_data = [comp_d.columns.tolist()] + comp_d.values.tolist()
t4 = Table(comp_data, colWidths=[3*cm, 3*cm, 3*cm, 3.5*cm, 4.2*cm])
t4_style = [
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#1a1a2e')),
    ('TEXTCOLOR',  (0,0), (-1,0), colors.white),
    ('FONTNAME',   (0,0), (-1,0), 'Helvetica-Bold'),
    ('FONTSIZE',   (0,0), (-1,-1), 8.5),
    ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.HexColor('#f5f5f5'), colors.white]),
    ('GRID',       (0,0), (-1,-1), 0.4, colors.HexColor('#cccccc')),
    ('ALIGN',      (0,0), (-1,-1), 'CENTER'),
    ('VALIGN',     (0,0), (-1,-1), 'MIDDLE'),
    ('TOPPADDING', (0,0), (-1,-1), 4),
    ('BOTTOMPADDING', (0,0), (-1,-1), 4),
]
for i, row in enumerate(comp_d.values.tolist()):
    if '3/3' in str(row[-1]):
        t4_style.append(('BACKGROUND', (4, i+1), (4, i+1), colors.HexColor('#d4edda')))
    elif 'discordante' in str(row[-1]):
        t4_style.append(('BACKGROUND', (4, i+1), (4, i+1), colors.HexColor('#fff3cd')))
t4.setStyle(TableStyle(t4_style))
story += [t4, SP(14)]

story += [
    HR(),
    H1('4.  Conclusions and Recommendations'),
    H2('4.1  Which tool performs best?'),
    P('<b>AtaCNV is the recommended tool</b> for CNA inference from scATAC-seq data '
      'in this context. After PCR duplicate correction, it achieves the best balance '
      'between sensitivity and specificity for gain detection '
      '(sensitivity 0.870, specificity 0.791) compared to the CCLE ground truth.'),
    P('epiAneufinder achieves the same sensitivity for gains (0.870) but at the '
      'cost of a much higher false positive rate (specificity 0.581) — it tends to '
      'call gain even in regions that are truly neutral. Think of it as a very sensitive '
      'smoke alarm that also goes off when you make toast.'),
    P('CONGASp also improved after correction but still performs poorly '
      'at 1 Mb resolution (sensitivity 0.314 for gain, specificity 0.666). '
      'Its Bayesian model is likely better suited to coarser segmentation or to '
      'datasets with more cells and deeper sequencing.'),
    H2('4.2  Which CNA regions are most reliable?'),
    P('The CNA calls supported by all three tools simultaneously are the most '
      'trustworthy. These are:'),
    P('&nbsp;&nbsp;&nbsp;&nbsp;<b>Gain: chr8, chr19</b> — amplification detected by '
      'AtaCNV and epiAneufinder (2 out of 3 tools). CONGASp systematically '
      'calls loss genome-wide, likely due to model miscalibration at 1 Mb resolution.'),
    P('&nbsp;&nbsp;&nbsp;&nbsp;<b>Loss: chr9, chr18, chr21</b> — deletion consistently '
      'detected by all three tools (3/3). These are the most reliable CNA calls in DV90.'),
    H2('4.3  Important caveats'),
    P('The CCLE ground truth is derived from <b>bulk</b> DNA sequencing — it represents '
      'the average copy number across millions of cells, not individual cells. '
      'Rare subclonal CNAs present in only a small fraction of cells may not be '
      'captured by CCLE, even if the scATAC tools detect them correctly.'),
    P('The dataset is highly class-imbalanced: <b>91% of 1 Mb bins are neutral</b> '
      '(CN=2) according to CCLE. This makes overall accuracy a misleading metric — '
      'a tool that always calls neutral would be right 91% of the time. '
      'This is why we focus on sensitivity and specificity separately.'),
    H2('4.4  Next steps'),
    P('Having selected AtaCNV as the best-performing tool, the next phase of the '
      'project will determine the <b>minimum sequencing depth</b> required for '
      'reliable CNA calling. This will be done by randomly downsampling the DV90 '
      'fragment file to 67%, 50%, 33% and 25% of the original depth, running '
      'AtaCNV on each subsample 100 times (with different random seeds), and '
      'comparing performance against the CCLE ground truth at each coverage level. '
      'This analysis is currently running in parallel on the server.'),
]

doc.build(story)
print(f'Report saved: {PDF_OUT}')

Report saved: /sharedFolder/Results/DV90_CNA_Benchmarking_Report_v3.pdf
